# Redrob Intelligent Candidate Ranker — Reproducibility Demo

**Challenge spec §10.5 sandbox demo.**

This notebook runs the **identical code** used for the full 100 K submission on a
200-candidate slice (first 200 rows of `candidates.jsonl`). It demonstrates the
complete pipeline — feature extraction, BM25 indexing, embedding, hybrid retrieval,
structural scoring, and reasoning generation — and produces a validator-passing
ranked CSV.

| | Full run | This demo |
|---|---|---|
| Candidate pool | 100,000 | 200 (first 200 from `candidates.jsonl`) |
| Output rows | 100 | 100 (top-100 of the 200-candidate pool) |
| Embedding model | all-MiniLM-L6-v2 | same |
| Code modified? | — | no |
| Runtime | ~35 min (precompute) + 5 s (rank) | **< 4 min total** |

> **"Run all"** on a fresh Colab CPU runtime. No GPU, no data upload required.

In [ ]:
# ── 1. Clone repo & install dependencies ─────────────────────────────────────
!git clone https://github.com/HARJAPAN2005/india-runs-candidate-ranker.git
%cd india-runs-candidate-ranker
!pip install -q -r requirements.txt
print('\nRepo cloned and dependencies installed.')

In [ ]:
# ── 2. Prepare 200-candidate sample ──────────────────────────────────────────
# demo_candidates.json (committed to repo) is the first 200 rows of the full
# candidates.jsonl. precompute.py expects one JSON object per line (JSONL).
import json

src = 'India_runs_data_and_ai_challenge/demo_candidates.json'
dst = 'India_runs_data_and_ai_challenge/candidates.jsonl'

with open(src, encoding='utf-8') as f:
    candidates = json.load(f)

with open(dst, 'w', encoding='utf-8') as f:
    for c in candidates:
        f.write(json.dumps(c) + '\n')

# Quick sanity check on expected gate behaviour
yoe_gte5 = sum(
    1 for c in candidates
    if sum(r.get('duration_months', 0) for r in c.get('career_history', [])) / 12 >= 5
)
print(f'Demo pool   : {len(candidates)} candidates')
print(f'yoe >= 5    : {yoe_gte5}  (will be eligible for ranking)')
print(f'yoe < 5     : {len(candidates) - yoe_gte5}  (will be gated out by yoe floor)')
print(f'Written to  : {dst}')

In [ ]:
# ── 3. Build artifacts ────────────────────────────────────────────────────────
# Downloads all-MiniLM-L6-v2 (~86 MB) on first run — network is available here.
# On 200 candidates: feature extraction ~10 s, embedding ~30 s. Total ~2-3 min.
!python precompute.py

In [ ]:
# ── 4. Rank candidates ────────────────────────────────────────────────────────
# Zero network calls at this step (TRANSFORMERS_OFFLINE=1, local_files_only=True).
# Outputs submission.csv with 100 ranked candidates.
!python rank.py

In [ ]:
# ── 5. Display ranked output ──────────────────────────────────────────────────
import pandas as pd

sub = pd.read_csv('submission.csv')
print(f'submission.csv: {len(sub)} rows')
print(f'Score range   : {sub["score"].min():.4f} – {sub["score"].max():.4f}')
print()

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)
sub[['rank', 'candidate_id', 'score', 'reasoning']]

In [ ]:
# ── 6. Validate submission ────────────────────────────────────────────────────
!python India_runs_data_and_ai_challenge/validate_submission.py submission.csv